# 多层感知机的简洁实现
:label:`sec_mlp_concise`

本节将介绍(**通过高级API更简洁地实现多层感知机**)。


In [2]:
import torch  # PyTorch深度学习框架
from torch import nn  # 神经网络模块
from d2l import torch as d2l  # 动手学深度学习工具包

## 模型

与softmax回归的简洁实现（ :numref:`sec_softmax_concise`）相比，
唯一的区别是我们添加了2个全连接层（之前我们只添加了1个全连接层）。
第一层是[**隐藏层**]，它(**包含256个隐藏单元，并使用了ReLU激活函数**)。
第二层是输出层。


* 256：一次输入256张图片（隐藏层，人为设置的超参数，还可以改为128，512等）
* 1：灰度图片只有一个通道
* 28：图像高度
* 28：图像宽度

In [9]:
# 使用Sequential按顺序构建多层感知机
net = nn.Sequential(
    nn.Flatten(),          # 将28×28图像展平为784维向量
    nn.Linear(784, 256),   # 隐藏层：784个输入，256个神经元
    nn.ReLU(),             # ReLU激活函数，引入非线性
    nn.Linear(256, 10)     # 输出层：256个输入，10个类别
)

def init_weights(m):
    """初始化全连接层权重"""
    if type(m) == nn.Linear:  # 判断是否为全连接层
        nn.init.normal_(m.weight, std=0.01)  # 正态分布初始化权重

net.apply(init_weights)  # 对模型各层应用初始化函数

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=10, bias=True)
)

[**训练过程**]的实现与我们实现softmax回归时完全相同，
这种模块化设计使我们能够将与模型架构有关的内容独立出来。


In [5]:
# 设置批量大小、学习率和训练轮数
batch_size, lr, num_epochs = 256, 0.1, 10

# 定义交叉熵损失，保留每个样本的损失值
loss = nn.CrossEntropyLoss(reduction='none')

# 使用SGD优化模型所有可训练参数
trainer = torch.optim.SGD(net.parameters(), lr=lr)

In [7]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer):
    """训练多层感知机模型"""
    for epoch in range(num_epochs):
        net.train()  # 训练模式
        train_loss, train_correct, train_total = 0, 0, 0

        for X, y in train_iter:
            y_hat = net(X)       # 前向传播
            l = loss(y_hat, y)   # 计算损失

            trainer.zero_grad()  # 清空梯度
            l.mean().backward()  # 反向传播
            trainer.step()       # 更新参数

            train_loss += l.sum().item()
            train_correct += (y_hat.argmax(1) == y).sum().item()
            train_total += y.numel()

        # 计算测试集准确率
        net.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for X, y in test_iter:
                correct += (net(X).argmax(1) == y).sum().item()
                total += y.numel()

        print(f"Epoch {epoch+1}: "
              f"loss={train_loss/train_total:.4f}, "
              f"train_acc={train_correct/train_total:.4f}, "
              f"test_acc={correct/total:.4f}")

In [8]:
# 加载Fashion-MNIST训练集和测试集
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

# 训练多层感知机模型
# train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

Epoch 1: loss=1.0476, train_acc=0.6444, test_acc=0.7041
Epoch 2: loss=0.6014, train_acc=0.7884, test_acc=0.7924
Epoch 3: loss=0.5191, train_acc=0.8191, test_acc=0.8222
Epoch 4: loss=0.4803, train_acc=0.8312, test_acc=0.8277
Epoch 5: loss=0.4537, train_acc=0.8406, test_acc=0.8363
Epoch 6: loss=0.4326, train_acc=0.8470, test_acc=0.8327
Epoch 7: loss=0.4184, train_acc=0.8514, test_acc=0.8174
Epoch 8: loss=0.4049, train_acc=0.8563, test_acc=0.8473
Epoch 9: loss=0.3913, train_acc=0.8617, test_acc=0.8255
Epoch 10: loss=0.3822, train_acc=0.8638, test_acc=0.8394


输入层：784
    ↓
全连接层：784 → 256
    ↓
ReLU 激活函数
    ↓
全连接层：256 → 10
    ↓
输出：10 个类别的预测分数

## 小结

* 我们可以使用高级API更简洁地实现多层感知机。
* 对于相同的分类问题，多层感知机的实现与softmax回归的实现相同，只是多层感知机的实现里增加了带有激活函数的隐藏层。

[Discussions](https://discuss.d2l.ai/t/1802)
